## 1. Import Libraries
Import all required Python libraries for data analysis, geospatial calculations, and visualization.

In [1]:
import pandas as pd
import numpy as np
import altair as alt
from geopy.distance import geodesic

## 2. Data Loading Functions
Define functions to load food security, port, and centroid data.

In [2]:
def load_data():
    # 1. Food Security Data
    food_df = pd.read_csv("data/undernourishment_raw.csv")
    # Process the raw World Bank export
    year_cols = [c for c in food_df.columns if c.startswith('YR')]
    food_df['latest_undernourishment'] = food_df[year_cols].bfill(axis=1).iloc[:, 0]
    food_df = food_df[['Country', 'latest_undernourishment']]
    print(food_df.head())
    # 2. Major Ports Data
    ports_df = pd.read_csv("data/major_ports.csv")
    # 3. Country Centroids Data
    centroids_df = pd.read_csv("data/country_centroids.csv")
    return food_df, ports_df, centroids_df

## 3. Distance Calculation Function
Define a function to calculate the distance from each country to its nearest major port.

In [3]:
def calculate_nearest_port_distance(country_df, ports_df):
    def get_min_dist(row):
        # Coordinates in country_centroids.csv are 'latitude' and 'longitude'
        country_coord = (row['latitude'], row['longitude'])
        # Coordinates in major_ports.csv are 'lat' and 'lon'
        distances = [geodesic(country_coord, (p['lat'], p['lon'])).km for _, p in ports_df.iterrows()]
        return min(distances)
    country_df['dist_to_nearest_port_km'] = country_df.apply(get_min_dist, axis=1)
    return country_df

## 4. Load Data
Load the food security, ports, and centroid data using the defined function.

In [4]:
try:
    food_df, ports_df, centroids_df = load_data()
except FileNotFoundError as e:
    print(f"Error: {e}. Please ensure the 'data' folder contains the required CSV files.")

                 Country  latest_undernourishment
0               Zimbabwe                     30.0
1                 Zambia                     32.1
2            Yemen, Rep.                      NaN
3     West Bank and Gaza                      NaN
4  Virgin Islands (U.S.)                      NaN


## 5. Merge DataFrames
Merge the food security data with country centroids for further analysis.

In [5]:
# Note: Using 'left' merge to keep all food data, then dropping countries without coordinates
merged_df = pd.merge(food_df, centroids_df, left_on='Country', right_on='name')

## 6. Calculate Distance to Nearest Port
Add a column for the distance from each country to its nearest major port.

In [6]:
final_df = calculate_nearest_port_distance(merged_df, ports_df)

## 7. Visualize Results
Create a scatter plot showing the relationship between distance to port and undernourishment.

In [7]:
chart = alt.Chart(final_df).mark_point(filled=True, size=60).encode(
    x=alt.X('dist_to_nearest_port_km:Q', title='Distance to Nearest Major Port (km)'),
    y=alt.Y('latest_undernourishment:Q', title='Prevalence of Undernourishment (%)'),
    color=alt.Color('dist_to_nearest_port_km:Q', scale=alt.Scale(scheme='viridis')),
    tooltip=['Country', 'dist_to_nearest_port_km', 'latest_undernourishment']
).properties(
    width=600,
    height=400,
    title='Impact of Supply Chain Proximity on Food Security'
).interactive()
chart

alt.Chart(...)

## 8. Show and Save Processed Data
Display a sample of the processed data and save the results to a CSV file.

In [8]:
print("\nSample of Processed Data:")
display(final_df[['Country', 'dist_to_nearest_port_km', 'latest_undernourishment']].head())

# Save the final merged dataset for further analysis
output_file = 'processed_food_security_analysis.csv'
final_df.to_csv(output_file, index=False)
print(f"\nAnalysis ready! Results saved to '{output_file}'")


Sample of Processed Data:


,Country,dist_to_nearest_port_km,latest_undernourishment
0,Zimbabwe,603.957118,30.0
1,Zambia,1052.148811,32.1
2,Vanuatu,1966.503191,7.9
3,Uzbekistan,1845.476744,2.5
4,Uruguay,266.371814,2.5



Analysis ready! Results saved to 'processed_food_security_analysis.csv'
